In [6]:
! pip install pandas
! pip install numpy 
! pip install torch 
! pip install transformers 
! pip install tqdm 
! pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


...


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# =========================================================
# TikTok Sentiment + Inferred Stars using CAMeLBERT
# =========================================================

# Install first if needed:
# pip install transformers torch pandas openpyxl tqdm

import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==============================
# 1) Settings
# ==============================
INPUT_FILE = r"C:\Users\ziyad\OneDrive\Desktop\EDA\1_textready1.xlsx"
OUTPUT_FILE = "tiktok_camelbert_labeled.xlsx"

# Prefer your cleaned text column
TEXT_CANDIDATES = ["Text_TR", "Text_ML", "Text", "text"]

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
BATCH_SIZE = 64
MAX_LENGTH = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ==============================
# 2) Load data
# ==============================
df = pd.read_excel(INPUT_FILE)

text_col = None
for c in TEXT_CANDIDATES:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"No text column found. Checked: {TEXT_CANDIDATES}")

print("Selected text column:", text_col)

# Clean nulls
df[text_col] = df[text_col].fillna("").astype(str).str.strip()

# ==============================
# 3) Load model
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ==============================
# 4) Label mapping
# ==============================
# We read the model labels dynamically in case order differs
id2label_raw = model.config.id2label
print("Raw model labels:", id2label_raw)

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).lower().strip()
    if "pos" in lbl:
        return "positive"
    if "neg" in lbl:
        return "negative"
    if "neu" in lbl:
        return "neutral"
    return lbl

id2label = {int(k): normalize_label(v) for k, v in id2label_raw.items()}

# Reverse lookup for probability extraction
label2id = {v: k for k, v in id2label.items()}

required = {"positive", "negative", "neutral"}
if not required.issubset(set(label2id.keys())):
    raise ValueError(
        f"Expected sentiment labels {required}, but got {set(label2id.keys())}"
    )

# ==============================
# 5) Star inference logic
# ==============================
def infer_stars(pos_prob, neu_prob, neg_prob):
    """
    Convert sentiment probabilities to inferred stars.
    These are inferred stars, not real user ratings.
    """
    probs = {
        "positive": pos_prob,
        "neutral": neu_prob,
        "negative": neg_prob
    }
    top_label = max(probs, key=probs.get)
    top_prob = probs[top_label]

    # More conservative mapping:
    # very confident positive -> 5
    # positive -> 4
    # neutral -> 3
    # negative -> 2
    # very confident negative -> 1
    if top_label == "positive":
        return 5 if top_prob >= 0.85 else 4
    elif top_label == "neutral":
        return 3
    else:
        return 1 if top_prob >= 0.85 else 2

# Optional: Arabic display label
def to_arabic_label(label):
    mapping = {
        "positive": "إيجابي",
        "neutral": "محايد",
        "negative": "سلبي"
    }
    return mapping.get(label, label)

# ==============================
# 6) Batch inference
# ==============================
all_pred_labels = []
all_pred_labels_ar = []
all_confidences = []
all_pos_probs = []
all_neu_probs = []
all_neg_probs = []
all_inferred_stars = []

texts = df[text_col].tolist()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting"):
    batch_texts = texts[start:start + BATCH_SIZE]

    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    pred_ids = probs.argmax(axis=1)

    for i, pred_id in enumerate(pred_ids):
        pred_label = id2label[int(pred_id)]
        confidence = float(probs[i][pred_id])

        pos_prob = float(probs[i][label2id["positive"]])
        neu_prob = float(probs[i][label2id["neutral"]])
        neg_prob = float(probs[i][label2id["negative"]])

        stars = infer_stars(pos_prob, neu_prob, neg_prob)

        all_pred_labels.append(pred_label)
        all_pred_labels_ar.append(to_arabic_label(pred_label))
        all_confidences.append(round(confidence, 4))
        all_pos_probs.append(round(pos_prob, 4))
        all_neu_probs.append(round(neu_prob, 4))
        all_neg_probs.append(round(neg_prob, 4))
        all_inferred_stars.append(stars)

# ==============================
# 7) Save results
# ==============================
df["Emotion_Label"] = all_pred_labels
df["Emotion_Label_AR"] = all_pred_labels_ar
df["Confidence"] = all_confidences
df["Prob_Positive"] = all_pos_probs
df["Prob_Neutral"] = all_neu_probs
df["Prob_Negative"] = all_neg_probs
df["Inferred_Stars"] = all_inferred_stars

df.to_excel(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")

# ==============================
# 8) Quick summary
# ==============================
print("\nEmotion distribution:")
print(df["Emotion_Label"].value_counts(dropna=False))

print("\nInferred stars distribution:")
print(df["Inferred_Stars"].value_counts(dropna=False).sort_index())

Using device: cpu
Selected text column: Text_TR


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

C:\Users\ziyad\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ziyad\.cache\huggingface\hub\models--CAMeL-Lab--bert-base-arabic-camelbert-da-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Raw model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


Predicting:   0%|          | 0/21 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Saved to: tiktok_camelbert_labeled.xlsx

Emotion distribution:
Emotion_Label
neutral     541
negative    474
positive    288
Name: count, dtype: int64

Inferred stars distribution:
Inferred_Stars
1    233
2    241
3    541
4    120
5    168
Name: count, dtype: int64


In [3]:
df = pd.read_excel("tiktok_camelbert_labeled.xlsx")
df.head()

,cid,createTime,createTimeISO,detailedMentions/0/nickName,detailedMentions/0/profileUrl,detailedMentions/0/secUid,detailedMentions/0/userId,detailedMentions/1/nickName,detailedMentions/1/profileUrl,detailedMentions/1/secUid,...,Text_ML,Is_Short_TR,Is_Short_ML,Emotion_Label,Emotion_Label_AR,Confidence,Prob_Positive,Prob_Neutral,Prob_Negative,Inferred_Stars
0,7307383077495243776,1701382766,2023-11-30T22:19:26.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,للاسف خيرنا لغيرنا EMO_NEU,False,False,negative,سلبي,0.9166,0.0448,0.0385,0.9166,1
1,7307398009677956096,1701386222,2023-11-30T23:17:02.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,الي بيروح العلا ويجرب مغامرات يروح مركز مررا ر...,False,False,neutral,محايد,0.7576,0.2378,0.7576,0.0046,3
2,7307379630226408448,1701381989,2023-11-30T22:06:29.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,طبعا غير الفطور والغداء والعشاء قضيها تصبيرة ط...,False,False,negative,سلبي,0.5543,0.1094,0.3363,0.5543,2
3,7307380227284993024,1701382105,2023-11-30T22:08:25.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,لا_تقدر تاكل الي ودك فيه المطاعم الغالية واللذ...,False,False,negative,سلبي,0.9494,0.0098,0.0409,0.9494,1
4,7433024184849171456,1730635811,2024-11-03T12:10:11.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,رحت لها وميزانيتي متوسطه وتجربه فنانه EMO_NEU,False,False,positive,إيجابي,0.8913,0.8913,0.0629,0.0458,5


In [4]:
import re
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==============================
# 1) Settings
# ==============================
INPUT_FILE = r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الجنوبية ) TikTok Data -  After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 1_textready.xlsx"
OUTPUT_FILE = r"C:\Users\ziyad\OneDrive\Desktop\EDA\tiktok_camelbert_labeled2.xlsx"

TEXT_CANDIDATES = ["Text_TR", "Text_ML", "Text", "text"]
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
BATCH_SIZE = 32
MAX_LENGTH = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ==============================
# 2) Load data
# ==============================
df = pd.read_excel(INPUT_FILE)

text_col = None
for c in TEXT_CANDIDATES:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"No text column found. Checked: {TEXT_CANDIDATES}")

print("Selected text column:", text_col)

df[text_col] = df[text_col].fillna("").astype(str).str.strip()

# ==============================
# 3) Load model
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ==============================
# 4) Label mapping
# ==============================
id2label_raw = model.config.id2label

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).lower().strip()
    if "pos" in lbl:
        return "positive"
    if "neg" in lbl:
        return "negative"
    if "neu" in lbl:
        return "neutral"
    return lbl

id2label = {int(k): normalize_label(v) for k, v in id2label_raw.items()}
label2id = {v: k for k, v in id2label.items()}

required = {"positive", "negative", "neutral"}
if not required.issubset(set(label2id.keys())):
    raise ValueError(f"Expected labels {required}, but got {set(label2id.keys())}")

# ==============================
# 5) Heuristic refinement
# ==============================
QUESTION_WORDS = [
    "هل", "وش", "ايش", "كيف", "ليه", "لِماذا", "متى", "وين", "بكم",
    "كم", "طيب", "يعني", "الحين", "أفهم", "بفهم", "ممكن", "؟", "?"
]

POSITIVE_HINTS = [
    "جميل", "رائع", "ممتاز", "حلو", "حلوه", "روعة", "يعجب", "احب",
    "مبسوط", "ممتعه", "مميز", "فخم", "ابداع", "يستاهل", "نورت"
]

NEGATIVE_HINTS = [
    "سيء", "سيئ", "خايس", "مقلب", "نصب", "تعب", "زحمة", "غالي", "غالي جدا",
    "سيئة", "قذر", "مزعج", "ما انصح", "سيء جدا", "فاشل", "شين"
]

def is_question_like(text: str) -> bool:
    text = str(text).strip().lower()
    if not text:
        return False

    # Contains question mark
    if "?" in text or "؟" in text:
        return True

    # Starts with or contains strong question cues
    for w in QUESTION_WORDS:
        if re.search(rf"\b{re.escape(w.lower())}\b", text):
            return True

    # Arabic colloquial question pattern
    patterns = [
        r"^طيب\b",
        r"\bبفهم\b",
        r"\bيعني\b",
        r"\bهل\b",
        r"\bوش\b",
        r"\bايش\b",
        r"\bكيف\b",
        r"\bكم\b",
        r"\bبكم\b",
        r"\bولا باقي\b",
    ]
    return any(re.search(p, text) for p in patterns)

def has_clear_positive(text: str) -> bool:
    text = str(text).lower()
    return any(w in text for w in POSITIVE_HINTS)

def has_clear_negative(text: str) -> bool:
    text = str(text).lower()
    return any(w in text for w in NEGATIVE_HINTS)

def refine_prediction(text, pred_label, pos_prob, neu_prob, neg_prob):
    """
    Improve prediction for question-like and ambiguous comments.
    """
    text = str(text).strip()

    # 1) Question-like comments with no clear praise/complaint -> neutral
    if is_question_like(text):
        if not has_clear_positive(text) and not has_clear_negative(text):
            return "neutral"

    # 2) If model says negative but negative score is not much stronger than neutral,
    # and the comment looks like inquiry -> neutral
    if pred_label == "negative":
        if is_question_like(text) and (neg_prob - neu_prob) < 0.20:
            return "neutral"

    # 3) If model says positive but no clear positive cue and neutral is close -> neutral
    if pred_label == "positive":
        if not has_clear_positive(text) and (pos_prob - neu_prob) < 0.15:
            return "neutral"

    return pred_label

def map_stars(sentiment):
    # inferred stars only
    if sentiment == "positive":
        return 5
    elif sentiment == "neutral":
        return 3
    else:
        return 1

# ==============================
# 6) Batch inference
# ==============================
sentiments = []
stars_list = []

texts = df[text_col].tolist()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting"):
    batch_texts = texts[start:start + BATCH_SIZE]

    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    pred_ids = probs.argmax(axis=1)

    for i, pred_id in enumerate(pred_ids):
        raw_label = id2label[int(pred_id)]

        pos_prob = float(probs[i][label2id["positive"]])
        neu_prob = float(probs[i][label2id["neutral"]])
        neg_prob = float(probs[i][label2id["negative"]])

        final_label = refine_prediction(
            text=batch_texts[i],
            pred_label=raw_label,
            pos_prob=pos_prob,
            neu_prob=neu_prob,
            neg_prob=neg_prob
        )

        sentiments.append(final_label)
        stars_list.append(map_stars(final_label))

# ==============================
# 7) Keep only required columns
# ==============================
df["Sentiment"] = sentiments
df["Stars"] = stars_list

# إذا تبغى الملف النهائي يحتوي كل الأعمدة الأصلية + العمودين:
# df.to_excel(OUTPUT_FILE, index=False)

# إذا تبغاه فقط النص + العمودين:
final_df = df[[text_col, "Sentiment", "Stars"]].copy()
final_df.to_excel(OUTPUT_FILE, index=False)

print(f"Saved to: {OUTPUT_FILE}")
print("\nSentiment distribution:")
print(final_df["Sentiment"].value_counts(dropna=False))

print("\nStars distribution:")
print(final_df["Stars"].value_counts(dropna=False).sort_index())

Using device: cpu
Selected text column: Text_TR


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicting:   0%|          | 0/6 [00:00<?, ?it/s]

Saved to: C:\Users\ziyad\OneDrive\Desktop\EDA\tiktok_camelbert_labeled2.xlsx

Sentiment distribution:
Sentiment
positive    69
neutral     68
negative    37
Name: count, dtype: int64

Stars distribution:
Stars
1    37
3    68
5    69
Name: count, dtype: int64


In [5]:
df = pd.read_excel("tiktok_camelbert_labeled2.xlsx")
df.head()

,Text_TR,Sentiment,Stars
0,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,neutral,3
1,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,positive,5
2,مرحبا بك بين اهلك وناسك من زهران وغامد واحد من...,positive,5
3,نورت بلاد الزهارين,positive,5
4,الحمدلله علي نعمه الباحه ياخي يوم تدخلها وتشوف...,positive,5


In [9]:
import re
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==============================
# 1) Settings
# ==============================
INPUT_FILE = r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الجنوبية ) TikTok Data -  After Cleaning\الأماكن السياحية Tik Tok - منطقة الباحة\Video comments 1_textready.xlsx"
OUTPUT_FILE = r"C:\Users\ziyad\OneDrive\Desktop\EDA\tiktok_camelbert_labeled222.xlsx"

TEXT_CANDIDATES = ["Text_TR", "Text_ML", "Text", "text"]
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
BATCH_SIZE = 32
MAX_LENGTH = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ==============================
# 2) Load data
# ==============================
df = pd.read_excel(INPUT_FILE)

text_col = None
for c in TEXT_CANDIDATES:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"No text column found. Checked: {TEXT_CANDIDATES}")

print("Selected text column:", text_col)
df[text_col] = df[text_col].fillna("").astype(str).str.strip()

# ==============================
# 3) Load model
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ==============================
# 4) Label mapping
# ==============================
id2label_raw = model.config.id2label

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).lower().strip()
    if "pos" in lbl:
        return "positive"
    if "neg" in lbl:
        return "negative"
    if "neu" in lbl:
        return "neutral"
    return lbl

id2label = {int(k): normalize_label(v) for k, v in id2label_raw.items()}
label2id = {v: k for k, v in id2label.items()}

required = {"positive", "negative", "neutral"}
if not required.issubset(set(label2id.keys())):
    raise ValueError(f"Expected labels {required}, but got {set(label2id.keys())}")

print("Model labels:", id2label)

# ==============================
# 5) Light refinement only
# ==============================
QUESTION_WORDS = {
    "متى", "هل", "كم", "بكم", "كيف", "ليش", "ليه", "وش", "ايش", "وين"
}

NEUTRAL_HINTS = {
    "للتوضيح", "توضيح", "معلومة", "نصيحة", "يفضل", "الأفضل", "افضل"
}

CLEAR_NEGATIVE_WORDS = {
    "غالي", "زحمة", "سيء", "سيئ", "خايس", "مقلب", "نصب", "فاشل", "مزعج"
}

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def contains_word(text, words):
    for w in words:
        if re.search(rf"(?<!\w){re.escape(w)}(?!\w)", text):
            return True
    return False

def is_question_like(text):
    text = clean_text(text)
    if "؟" in text or "?" in text:
        return True
    if contains_word(text, QUESTION_WORDS):
        return True
    return False

def is_explanatory_or_advice(text):
    text = clean_text(text)
    if contains_word(text, NEUTRAL_HINTS):
        return True
    return False

def has_clear_negative(text):
    text = clean_text(text)
    if contains_word(text, CLEAR_NEGATIVE_WORDS):
        return True
    return False

def refine_prediction(text, model_label, pos_prob, neu_prob, neg_prob):
    text = clean_text(text)

    # 1) السؤال الصريح -> محايد
    if is_question_like(text):
        return "neutral"

    # 2) التوضيح / النصيحة -> محايد
    if is_explanatory_or_advice(text):
        return "neutral"

    # 3) الكلمات السلبية الواضحة جدًا -> سلبي
    if has_clear_negative(text):
        return "negative"

    # 4) إذا النموذج اختار positive أو negative لكن neutral قريب جدًا منه
    # في التعليقات القصيرة فقط، نخليه neutral
    wc = len(text.split())
    top_prob = max(pos_prob, neu_prob, neg_prob)

    if wc <= 4:
        if model_label in ["positive", "negative"] and abs(top_prob - neu_prob) < 0.10:
            return "neutral"

    return model_label

def map_stars(sentiment):
    if sentiment == "positive":
        return 5
    elif sentiment == "neutral":
        return 3
    else:
        return 1

# ==============================
# 6) Batch inference
# ==============================
sentiments = []
stars_list = []

texts = df[text_col].tolist()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting"):
    batch_texts = texts[start:start + BATCH_SIZE]

    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    pred_ids = probs.argmax(axis=1)

    for i, pred_id in enumerate(pred_ids):
        model_label = id2label[int(pred_id)]

        pos_prob = float(probs[i][label2id["positive"]])
        neu_prob = float(probs[i][label2id["neutral"]])
        neg_prob = float(probs[i][label2id["negative"]])

        final_label = refine_prediction(
            text=batch_texts[i],
            model_label=model_label,
            pos_prob=pos_prob,
            neu_prob=neu_prob,
            neg_prob=neg_prob
        )

        sentiments.append(final_label)
        stars_list.append(map_stars(final_label))

# ==============================
# 7) Add only new columns
# ==============================
df["Sentiment"] = sentiments
df["Stars"] = stars_list

# ==============================
# 8) Save
# ==============================
df.to_excel(OUTPUT_FILE, index=False)

print(f"Saved to: {OUTPUT_FILE}")

print("\nSentiment distribution:")
print(df["Sentiment"].value_counts(dropna=False))

print("\nStars distribution:")
print(df["Stars"].value_counts(dropna=False).sort_index())

Using device: cpu
Selected text column: Text_TR


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


Predicting:   0%|          | 0/6 [00:00<?, ?it/s]

Saved to: C:\Users\ziyad\OneDrive\Desktop\EDA\tiktok_camelbert_labeled222.xlsx

Sentiment distribution:
Sentiment
neutral     69
positive    69
negative    36
Name: count, dtype: int64

Stars distribution:
Stars
1    36
3    69
5    69
Name: count, dtype: int64


In [10]:
df = pd.read_excel("tiktok_camelbert_labeled222.xlsx")
df.head()

,text,diggCount,replyCommentTotal,createTimeISO,videoWebUrl,cid,Text_Orig,Text,Emoji_List,Emoji_Count,...,Emoji_Neg_Count,Emoji_Score,Emoji_Sentiment,Text_Base,Text_TR,Text_ML,Is_Short_TR,Is_Short_ML,Sentiment,Stars
0,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,163,6,2025-04-22T12:20:41.000Z,https://www.tiktok.com/@alshehhi277/video/7495...,7496111304841004032,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,[],0,...,0,0,NEU,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,المكان الي مصوره اكواخ الحازم الريفيع بمنطقة ا...,False,False,neutral,3
1,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,0,0,2025-10-28T08:46:15.000Z,https://www.tiktok.com/@alshehhi277/video/7495...,7566191184525673472,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,[],0,...,0,0,NEU,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,من اجمل الاماكن فعلا اكواخ الحازم تقع في المندق,اجمل الاماكن فعلا اكواخ الحازم تقع المندق EMO_NEU,False,False,positive,5
2,مرحباً بك بين اهلك وناسك من زهران وغامد واحد م...,32,1,2025-04-21T19:56:46.000Z,https://www.tiktok.com/@alshehhi277/video/7495...,7495857811416269824,مرحباً بك بين اهلك وناسك من زهران وغامد واحد م...,مرحباً بك بين اهلك وناسك من زهران وغامد واحد م...,['❤'],1,...,0,1,POS,مرحبا بك بين اهلك وناسك من زهران وغامد واحد من...,مرحبا بك بين اهلك وناسك من زهران وغامد واحد من...,مرحبا بك اهلك وناسك زهران وغامد واحد منا ولانت...,False,False,positive,5
3,نورت بلاد الزهارين ♥️♥️,7,0,2025-04-22T23:01:01.000Z,https://www.tiktok.com/@alshehhi277/video/7495...,7496276214971057152,نورت بلاد الزهارين ♥️♥️,نورت بلاد الزهارين ♥️♥️,"['♥', '♥']",2,...,0,0,NEU,نورت بلاد الزهارين,نورت بلاد الزهارين,نورت بلاد الزهارين EMO_NEU,False,False,positive,5
4,الحمدلله على نعمه الباحه ياخي يوم تدخلها وتشوف...,36,0,2025-04-23T15:25:27.000Z,https://www.tiktok.com/@alshehhi277/video/7495...,7496530027472880640,الحمدلله على نعمه الباحه ياخي يوم تدخلها وتشوف...,الحمدلله على نعمه الباحه ياخي يوم تدخلها وتشوف...,['❤'],1,...,0,1,POS,الحمدلله علي نعمه الباحه ياخي يوم تدخلها وتشوف...,الحمدلله علي نعمه الباحه ياخي يوم تدخلها وتشوف...,الحمدلله علي نعمه الباحه ياخي يوم تدخلها وتشوف...,False,False,positive,5


In [2]:
! pip install pandas
! pip instal torch
! pip install tqdm.auto
! pip install transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: unknown command "instal" - maybe you meant "install"



Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement tqdm.auto (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tqdm.auto


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import sys
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================
# 1) SETTINGS
# =========================================
ROOT_FOLDER = r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning"
OUTPUT_ROOT = r"C:\Users\ziyad\OneDrive\Desktop\EDA\ tiktok_labeled_output1.xlsx"

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
TEXT_CANDIDATES = ["Text_TR", "Text_ML", "Text", "text"]

BATCH_SIZE = 32
MAX_LENGTH = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SKIP_OUTPUT_FILES = True
OUTPUT_SUFFIX = ""   # نفس اسم الملف الأصلي
CSV_REGION_KEYWORD = "الوسطى"

AUTO_PROCEED = True  # إذا تبيه يوقف ويسألك خلها False

# =========================================
# 2) LOAD MODEL
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

id2label_raw = model.config.id2label

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).lower().strip()
    if "pos" in lbl:
        return "positive"
    if "neg" in lbl:
        return "negative"
    if "neu" in lbl:
        return "neutral"
    return lbl

id2label = {int(k): normalize_label(v) for k, v in id2label_raw.items()}
label2id = {v: k for k, v in id2label.items()}

required = {"positive", "negative", "neutral"}
if not required.issubset(set(label2id.keys())):
    raise ValueError(f"Expected labels {required}, but got {set(label2id.keys())}")

print("Model labels:", id2label)

# =========================================
# 3) LIGHT RULES
# =========================================
QUESTION_WORDS = {
    "متى", "هل", "كم", "بكم", "كيف", "ليش", "ليه", "وش", "ايش", "وين"
}

NEUTRAL_WORDS = {
    "للتوضيح", "توضيح", "نصيحة", "يفضل", "الأفضل", "افضل", "أنصح", "انصح"
}

CLEAR_NEGATIVE_WORDS = {
    "غالي", "زحمة", "سيء", "سيئ", "مزعج", "نصب", "مقلب", "فاشل"
}

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def contains_exact_word(text, words):
    for w in words:
        if re.search(rf"(?<!\w){re.escape(w)}(?!\w)", text):
            return True
    return False

def is_question_like(text):
    text = clean_text(text)
    if "؟" in text or "?" in text:
        return True
    return contains_exact_word(text, QUESTION_WORDS)

def is_neutral_hint(text):
    text = clean_text(text)
    return contains_exact_word(text, NEUTRAL_WORDS)

def has_clear_negative(text):
    text = clean_text(text)
    return contains_exact_word(text, CLEAR_NEGATIVE_WORDS)

def refine_prediction(text, model_label):
    text = clean_text(text)

    if not text:
        return "neutral"

    if is_question_like(text):
        return "neutral"

    if is_neutral_hint(text):
        return "neutral"

    if has_clear_negative(text):
        return "negative"

    return model_label

def map_stars(sentiment):
    if sentiment == "positive":
        return 5
    elif sentiment == "neutral":
        return 3
    else:
        return 1

# =========================================
# 4) FILE HELPERS
# =========================================
def is_output_file(filename_lower):
    return (
        filename_lower.endswith(f"{OUTPUT_SUFFIX}.xlsx")
        or filename_lower.endswith(f"{OUTPUT_SUFFIX}.xls")
        or filename_lower.endswith(f"{OUTPUT_SUFFIX}.csv")
    )

def should_include_file(file_path):
    filename = os.path.basename(file_path).lower()

    # بما أن OUTPUT_SUFFIX = ""، هذا الشرط ما عاد نحتاجه فعليًا
    if SKIP_OUTPUT_FILES and OUTPUT_SUFFIX != "" and is_output_file(filename):
        return False

    if filename.endswith(".xlsx") or filename.endswith(".xls"):
        return True

    if filename.endswith(".csv") and CSV_REGION_KEYWORD in file_path:
        return True

    return False

def read_table(file_path):
    lower = file_path.lower()

    if lower.endswith(".xlsx") or lower.endswith(".xls"):
        return pd.read_excel(file_path)

    if lower.endswith(".csv"):
        attempts = [
            {"encoding": "utf-8-sig", "sep": ","},
            {"encoding": "utf-8", "sep": ","},
            {"encoding": "cp1256", "sep": ","},
            {"encoding": "latin1", "sep": ","},
            {"encoding": "utf-16", "sep": ","},
            {"encoding": "utf-16", "sep": "\t"},
            {"encoding": "utf-8-sig", "sep": ";"},
            {"encoding": "utf-8", "sep": ";"},
            {"encoding": "cp1256", "sep": ";"},
            {"encoding": "latin1", "sep": ";"},
            {"encoding": "utf-16", "sep": ";"},
        ]

        last_error = None
        for attempt in attempts:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding=attempt["encoding"],
                    sep=attempt["sep"],
                    on_bad_lines="skip",
                    engine="python"
                )
                if df.shape[1] > 1:
                    return df
            except Exception as e:
                last_error = e

        raise ValueError(f"Failed to read CSV with all methods. Last error: {last_error}")

    raise ValueError("Unsupported file format.")

def build_output_path(original_path):
    rel_path = os.path.relpath(original_path, ROOT_FOLDER)
    output_path = os.path.join(OUTPUT_ROOT, rel_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path

def save_table(df, original_path):
    output_path = build_output_path(original_path)
    ext = os.path.splitext(output_path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df.to_excel(output_path, index=False)
    elif ext == ".csv":
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
    else:
        raise ValueError("Unsupported output format.")

    return output_path

def clean_missing_text_rows(df, chosen_text_col):
    before_rows = len(df)

    if "Text_TR" in df.columns:
        target_col = "Text_TR"
    else:
        target_col = chosen_text_col

    df = df[df[target_col].notna()].copy()
    df[target_col] = df[target_col].astype(str).str.strip()

    df = df[
        (df[target_col] != "") &
        (df[target_col].str.lower() != "nan")
    ].copy()

    removed_rows = before_rows - len(df)
    return df, target_col, removed_rows

# =========================================
# 5) FIND ALL FILES
# =========================================
all_files = []

for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        full_path = os.path.join(root, file)
        if should_include_file(full_path):
            all_files.append(full_path)

all_files = sorted(all_files)

ALL_FILES_SET = set(all_files)
TRACKED_FILES_SET = set()
PROCESSED_FILES_SET = set()

# =========================================
# 6) PRE-CHECK
# =========================================
print("\n" + "=" * 70)
print("PRE-CHECK: FILES FOUND")
print("=" * 70)
print(f"Total files found: {len(all_files)}\n")

if len(all_files) == 0:
    print("No files found. Check ROOT_FOLDER path.")
    sys.exit()

for i, file_path in enumerate(all_files[:30], start=1):
    print(f"{i}. {file_path}")

if len(all_files) > 30:
    print(f"\n... and {len(all_files) - 30} more files")

xlsx_count = sum(1 for f in all_files if f.lower().endswith(".xlsx"))
xls_count = sum(1 for f in all_files if f.lower().endswith(".xls"))
csv_count = sum(1 for f in all_files if f.lower().endswith(".csv"))

print("\nFile type summary:")
print(f".xlsx: {xlsx_count}")
print(f".xls : {xls_count}")
print(f".csv : {csv_count}")

if not AUTO_PROCEED:
    user_input = input("\nDo you want to proceed? (y/n): ").strip().lower()
    if user_input != "y":
        print("Process stopped by user.")
        sys.exit()

print("\nStarting processing...\n")

# =========================================
# 7) PROCESS FILES
# =========================================
processed_count = 0
skipped_count = 0
error_files = []
summary_rows = []

for file_path in tqdm(all_files, desc="Processing files"):
    TRACKED_FILES_SET.add(file_path)

    try:
        print(f"\nProcessing: {file_path}")

        # -------------------------
        # Read file
        # -------------------------
        df = read_table(file_path)
        print(f"  -> Loaded shape: {df.shape}")

        # -------------------------
        # Choose text column
        # -------------------------
        text_col = None
        for c in TEXT_CANDIDATES:
            if c in df.columns:
                text_col = c
                break

        if text_col is None:
            print("  -> Skipped: no text column found.")
            skipped_count += 1
            summary_rows.append({
                "file_path": file_path,
                "status": "skipped_no_text_column",
                "rows_before": len(df),
                "rows_removed": None,
                "rows_final": None,
                "output_path": None,
                "error": None
            })
            continue

        # -------------------------
        # Clean missing rows
        # -------------------------
        rows_before = len(df)
        df, text_col, removed_rows = clean_missing_text_rows(df, text_col)

        if len(df) == 0:
            print(f"  -> Skipped: all rows removed after dropping missing text. Removed {removed_rows} rows.")
            skipped_count += 1
            summary_rows.append({
                "file_path": file_path,
                "status": "skipped_all_rows_removed",
                "rows_before": rows_before,
                "rows_removed": removed_rows,
                "rows_final": 0,
                "output_path": None,
                "error": None
            })
            continue

        # -------------------------
        # Predict
        # -------------------------
        texts = df[text_col].astype(str).tolist()

        sentiments = []
        stars_list = []

        for start in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[start:start + BATCH_SIZE]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            with torch.no_grad():
                outputs = model(**enc)
                pred_ids = outputs.logits.argmax(dim=1).cpu().numpy()

            for i, pred_id in enumerate(pred_ids):
                model_label = id2label[int(pred_id)]

                final_label = refine_prediction(
                    text=batch_texts[i],
                    model_label=model_label
                )

                sentiments.append(final_label)
                stars_list.append(map_stars(final_label))

        # -------------------------
        # Add only two columns
        # -------------------------
        df["Sentiment"] = sentiments
        df["Stars"] = stars_list

        # -------------------------
        # Save in /kaggle/working with same structure
        # -------------------------
        output_path = save_table(df, file_path)

        print(f"  -> Saved: {output_path}")
        print(f"  -> Removed missing text rows: {removed_rows}")
        print(f"  -> Final rows: {len(df)}")

        processed_count += 1
        PROCESSED_FILES_SET.add(file_path)

        summary_rows.append({
            "file_path": file_path,
            "status": "processed",
            "rows_before": rows_before,
            "rows_removed": removed_rows,
            "rows_final": len(df),
            "output_path": output_path,
            "error": None
        })

    except Exception as e:
        print(f"  -> Error: {file_path}")
        print(f"     {e}")

        error_files.append((file_path, str(e)))
        summary_rows.append({
            "file_path": file_path,
            "status": "error",
            "rows_before": None,
            "rows_removed": None,
            "rows_final": None,
            "output_path": None,
            "error": str(e)
        })

# =========================================
# 8) SAVE SUMMARY
# =========================================
os.makedirs(OUTPUT_ROOT, exist_ok=True)
summary_df = pd.DataFrame(summary_rows)
summary_csv_path = os.path.join(OUTPUT_ROOT, "processing_summary.csv")
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

# =========================================
# 9) FINAL VERIFICATION
# =========================================
missing_tracked_files = ALL_FILES_SET - TRACKED_FILES_SET
missing_processed_files = ALL_FILES_SET - PROCESSED_FILES_SET

print("\n" + "=" * 70)
print("FINAL VERIFICATION CHECK")
print("=" * 70)
print(f"Total files found:        {len(ALL_FILES_SET)}")
print(f"Tracked in loop:          {len(TRACKED_FILES_SET)}")
print(f"Successfully processed:   {len(PROCESSED_FILES_SET)}")
print(f"Skipped files:            {skipped_count}")
print(f"Error files:              {len(error_files)}")
print(f"Untracked files:          {len(missing_tracked_files)}")
print(f"Not-successfully-processed files: {len(missing_processed_files)}")

if len(missing_tracked_files) == 0:
    print("\n✅ SUCCESS: The loop passed over all discovered files.")
else:
    print("\n❌ WARNING: Some discovered files were never entered in the loop:")
    for i, f in enumerate(list(missing_tracked_files)[:20], 1):
        print(f"{i}. {f}")
    if len(missing_tracked_files) > 20:
        print(f"... and {len(missing_tracked_files) - 20} more files")

print("\nSummary file saved to:")
print(summary_csv_path)

print("\nLabeled output root:")
print(OUTPUT_ROOT)

# =========================================
# 10) OPTIONAL: SHOW ERRORS
# =========================================
if error_files:
    print("\nFiles with errors:")
    for fp, err in error_files[:20]:
        print(f"- {fp}")
        print(f"  Error: {err}")
    if len(error_files) > 20:
        print(f"... and {len(error_files) - 20} more")

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}

PRE-CHECK: FILES FOUND
Total files found: 80

1. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-27-13-614_textready.xlsx
2. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-30-14-443_textready.xlsx
3. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-32-43-172_textready.xlsx
4. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-36-55-371_textready.xlsx
5. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok 

Processing files:   0%|          | 0/80 [00:00<?, ?it/s]


Processing: C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-27-13-614_textready.xlsx
  -> Loaded shape: (421, 19)
  -> Saved: C:\Users\ziyad\OneDrive\Desktop\EDA\ tiktok_labeled_output1.xlsx\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-27-13-614_textready.xlsx
  -> Removed missing text rows: 85
  -> Final rows: 336

Processing: C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) TikTok Data -  After Cleaning\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-30-14-443_textready.xlsx
  -> Loaded shape: (89, 19)
  -> Saved: C:\Users\ziyad\OneDrive\Desktop\EDA\ tiktok_labeled_output1.xlsx\الرياض\dataset_tiktok-comments-scraper_2025-11-05_15-30-14-443_textready.xlsx
  -> Removed missing text rows: 18
  -> Final rows: 71

Processing: C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From TikTok\( المنطقة الوسطى ) Tik

In [1]:
import os
import re
import sys
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================================
# 1) SETTINGS
# =========================================
ROOT_FOLDER = r"C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube"
OUTPUT_ROOT = r"C:\Users\ziyad\OneDrive\Desktop\EDA\youtube_labeled_output"

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
TEXT_CANDIDATES = ["Text_TR", "Text_ML", "Text", "comment", "text"]

BATCH_SIZE = 32
MAX_LENGTH = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SKIP_OUTPUT_FILES = True
OUTPUT_SUFFIX = ""
AUTO_PROCEED = True

# =========================================
# 2) LOAD MODEL
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

id2label_raw = model.config.id2label

def normalize_label(lbl: str) -> str:
    lbl = str(lbl).lower().strip()
    if "pos" in lbl:
        return "positive"
    if "neg" in lbl:
        return "negative"
    if "neu" in lbl:
        return "neutral"
    return lbl

id2label = {int(k): normalize_label(v) for k, v in id2label_raw.items()}
label2id = {v: k for k, v in id2label.items()}

required = {"positive", "negative", "neutral"}
if not required.issubset(set(label2id.keys())):
    raise ValueError(f"Expected labels {required}, but got {set(label2id.keys())}")

print("Model labels:", id2label)

# =========================================
# 3) LIGHT RULES
# =========================================
QUESTION_WORDS = {
    "متى", "هل", "كم", "بكم", "كيف", "ليش", "ليه", "وش", "ايش", "وين"
}

NEUTRAL_WORDS = {
    "للتوضيح", "توضيح", "نصيحة", "يفضل", "الأفضل", "افضل", "أنصح", "انصح"
}

CLEAR_NEGATIVE_WORDS = {
    "غالي", "زحمة", "سيء", "سيئ", "مزعج", "نصب", "مقلب", "فاشل"
}

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def contains_exact_word(text, words):
    for w in words:
        if re.search(rf"(?<!\w){re.escape(w)}(?!\w)", text):
            return True
    return False

def is_question_like(text):
    text = clean_text(text)
    if "؟" in text or "?" in text:
        return True
    return contains_exact_word(text, QUESTION_WORDS)

def is_neutral_hint(text):
    text = clean_text(text)
    return contains_exact_word(text, NEUTRAL_WORDS)

def has_clear_negative(text):
    text = clean_text(text)
    return contains_exact_word(text, CLEAR_NEGATIVE_WORDS)

def refine_prediction(text, model_label):
    text = clean_text(text)

    if not text:
        return "neutral"

    if is_question_like(text):
        return "neutral"

    if is_neutral_hint(text):
        return "neutral"

    if has_clear_negative(text):
        return "negative"

    return model_label

def map_stars(sentiment):
    if sentiment == "positive":
        return 5
    elif sentiment == "neutral":
        return 3
    else:
        return 1

# =========================================
# 4) FILE HELPERS
# =========================================
def should_include_file(file_path):
    filename = os.path.basename(file_path).lower()
    return filename.endswith(".xlsx") or filename.endswith(".xls") or filename.endswith(".csv")

def read_table(file_path):
    lower = file_path.lower()

    if lower.endswith(".xlsx") or lower.endswith(".xls"):
        return pd.read_excel(file_path)

    if lower.endswith(".csv"):
        attempts = [
            {"encoding": "utf-8-sig", "sep": ","},
            {"encoding": "utf-8", "sep": ","},
            {"encoding": "cp1256", "sep": ","},
            {"encoding": "latin1", "sep": ","},
            {"encoding": "utf-16", "sep": ","},
            {"encoding": "utf-16", "sep": "\t"},
            {"encoding": "utf-8-sig", "sep": ";"},
            {"encoding": "utf-8", "sep": ";"},
            {"encoding": "cp1256", "sep": ";"},
            {"encoding": "latin1", "sep": ";"},
            {"encoding": "utf-16", "sep": ";"},
        ]

        last_error = None
        for attempt in attempts:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding=attempt["encoding"],
                    sep=attempt["sep"],
                    on_bad_lines="skip",
                    engine="python"
                )
                if df.shape[1] > 1:
                    return df
            except Exception as e:
                last_error = e

        raise ValueError(f"Failed to read CSV with all methods. Last error: {last_error}")

    raise ValueError("Unsupported file format.")

def build_output_path(original_path):
    rel_path = os.path.relpath(original_path, ROOT_FOLDER)
    output_path = os.path.join(OUTPUT_ROOT, rel_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return output_path

def save_table(df, original_path):
    output_path = build_output_path(original_path)
    ext = os.path.splitext(output_path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df.to_excel(output_path, index=False)
    elif ext == ".csv":
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
    else:
        raise ValueError("Unsupported output format.")

    return output_path

def clean_missing_text_rows(df, chosen_text_col):
    before_rows = len(df)

    if "Text_TR" in df.columns:
        target_col = "Text_TR"
    else:
        target_col = chosen_text_col

    df = df[df[target_col].notna()].copy()
    df[target_col] = df[target_col].astype(str).str.strip()

    df = df[
        (df[target_col] != "") &
        (df[target_col].str.lower() != "nan")
    ].copy()

    removed_rows = before_rows - len(df)
    return df, target_col, removed_rows

# =========================================
# 5) FIND ALL FILES
# =========================================
all_files = []

for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        full_path = os.path.join(root, file)
        if should_include_file(full_path):
            all_files.append(full_path)

all_files = sorted(all_files)

ALL_FILES_SET = set(all_files)
TRACKED_FILES_SET = set()
PROCESSED_FILES_SET = set()

# =========================================
# 6) PRE-CHECK
# =========================================
print("\n" + "=" * 70)
print("PRE-CHECK: FILES FOUND")
print("=" * 70)
print(f"Total files found: {len(all_files)}\n")

if len(all_files) == 0:
    print("No files found. Check ROOT_FOLDER path.")
    sys.exit()

for i, file_path in enumerate(all_files[:30], start=1):
    print(f"{i}. {file_path}")

if len(all_files) > 30:
    print(f"\n... and {len(all_files) - 30} more files")

xlsx_count = sum(1 for f in all_files if f.lower().endswith(".xlsx"))
xls_count = sum(1 for f in all_files if f.lower().endswith(".xls"))
csv_count = sum(1 for f in all_files if f.lower().endswith(".csv"))

print("\nFile type summary:")
print(f".xlsx: {xlsx_count}")
print(f".xls : {xls_count}")
print(f".csv : {csv_count}")

if not AUTO_PROCEED:
    user_input = input("\nDo you want to proceed? (y/n): ").strip().lower()
    if user_input != "y":
        print("Process stopped by user.")
        sys.exit()

print("\nStarting processing...\n")

# =========================================
# 7) PROCESS FILES
# =========================================
processed_count = 0
skipped_count = 0
error_files = []
summary_rows = []

for file_path in tqdm(all_files, desc="Processing files"):
    TRACKED_FILES_SET.add(file_path)

    try:
        print(f"\nProcessing: {file_path}")

        # -------------------------
        # Read file
        # -------------------------
        df = read_table(file_path)
        print(f"  -> Loaded shape: {df.shape}")

        # -------------------------
        # Choose text column
        # -------------------------
        text_col = None
        for c in TEXT_CANDIDATES:
            if c in df.columns:
                text_col = c
                break

        if text_col is None:
            print("  -> Skipped: no text column found.")
            skipped_count += 1
            summary_rows.append({
                "file_path": file_path,
                "status": "skipped_no_text_column",
                "rows_before": len(df),
                "rows_removed": None,
                "rows_final": None,
                "output_path": None,
                "error": None
            })
            continue

        # -------------------------
        # Clean missing rows
        # -------------------------
        rows_before = len(df)
        df, text_col, removed_rows = clean_missing_text_rows(df, text_col)

        if len(df) == 0:
            print(f"  -> Skipped: all rows removed after dropping missing text. Removed {removed_rows} rows.")
            skipped_count += 1
            summary_rows.append({
                "file_path": file_path,
                "status": "skipped_all_rows_removed",
                "rows_before": rows_before,
                "rows_removed": removed_rows,
                "rows_final": 0,
                "output_path": None,
                "error": None
            })
            continue

        # -------------------------
        # Predict
        # -------------------------
        texts = df[text_col].astype(str).tolist()

        sentiments = []
        stars_list = []

        for start in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[start:start + BATCH_SIZE]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            with torch.no_grad():
                outputs = model(**enc)
                pred_ids = outputs.logits.argmax(dim=1).cpu().numpy()

            for i, pred_id in enumerate(pred_ids):
                model_label = id2label[int(pred_id)]

                final_label = refine_prediction(
                    text=batch_texts[i],
                    model_label=model_label
                )

                sentiments.append(final_label)
                stars_list.append(map_stars(final_label))

        # -------------------------
        # Add only two columns
        # -------------------------
        df["Sentiment"] = sentiments
        df["Stars"] = stars_list

        # -------------------------
        # Save with same folder structure
        # -------------------------
        output_path = save_table(df, file_path)

        print(f"  -> Saved: {output_path}")
        print(f"  -> Removed missing text rows: {removed_rows}")
        print(f"  -> Final rows: {len(df)}")

        processed_count += 1
        PROCESSED_FILES_SET.add(file_path)

        summary_rows.append({
            "file_path": file_path,
            "status": "processed",
            "rows_before": rows_before,
            "rows_removed": removed_rows,
            "rows_final": len(df),
            "output_path": output_path,
            "error": None
        })

    except Exception as e:
        print(f"  -> Error: {file_path}")
        print(f"     {e}")

        error_files.append((file_path, str(e)))
        summary_rows.append({
            "file_path": file_path,
            "status": "error",
            "rows_before": None,
            "rows_removed": None,
            "rows_final": None,
            "output_path": None,
            "error": str(e)
        })

# =========================================
# 8) SAVE SUMMARY
# =========================================
os.makedirs(OUTPUT_ROOT, exist_ok=True)
summary_df = pd.DataFrame(summary_rows)
summary_csv_path = os.path.join(OUTPUT_ROOT, "processing_summary.csv")
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

# =========================================
# 9) FINAL VERIFICATION
# =========================================
missing_tracked_files = ALL_FILES_SET - TRACKED_FILES_SET
missing_processed_files = ALL_FILES_SET - PROCESSED_FILES_SET

print("\n" + "=" * 70)
print("FINAL VERIFICATION CHECK")
print("=" * 70)
print(f"Total files found:              {len(ALL_FILES_SET)}")
print(f"Tracked in loop:                {len(TRACKED_FILES_SET)}")
print(f"Successfully processed:         {len(PROCESSED_FILES_SET)}")
print(f"Skipped files:                  {skipped_count}")
print(f"Error files:                    {len(error_files)}")
print(f"Untracked files:                {len(missing_tracked_files)}")
print(f"Not-successfully-processed:     {len(missing_processed_files)}")

if len(missing_tracked_files) == 0:
    print("\n✅ SUCCESS: The loop passed over all discovered files.")
else:
    print("\n❌ WARNING: Some discovered files were never entered in the loop:")
    for i, f in enumerate(list(missing_tracked_files)[:20], 1):
        print(f"{i}. {f}")
    if len(missing_tracked_files) > 20:
        print(f"... and {len(missing_tracked_files) - 20} more files")

print("\nSummary file saved to:")
print(summary_csv_path)

print("\nLabeled output root:")
print(OUTPUT_ROOT)

# =========================================
# 10) OPTIONAL: SHOW ERRORS
# =========================================
if error_files:
    print("\nFiles with errors:")
    for fp, err in error_files[:20]:
        print(f"- {fp}")
        print(f"  Error: {err}")
    if len(error_files) > 20:
        print(f"... and {len(error_files) - 20} more")

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}

PRE-CHECK: FILES FOUND
Total files found: 199

1. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 10_textready.xlsx
2. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 11_textready.xlsx
3. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 12_textready.xlsx
4. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 13_textready.xlsx
5. C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Clean

Processing files:   0%|          | 0/199 [00:00<?, ?it/s]


Processing: C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 10_textready.xlsx
  -> Loaded shape: (128, 19)
  -> Saved: C:\Users\ziyad\OneDrive\Desktop\EDA\youtube_labeled_output\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 10_textready.xlsx
  -> Removed missing text rows: 3
  -> Final rows: 125

Processing: C:\Users\ziyad\OneDrive\Desktop\EDA\DataSet\Cleaned Data From Youtube\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 11_textready.xlsx
  -> Loaded shape: (114, 19)
  -> Saved: C:\Users\ziyad\OneDrive\Desktop\EDA\youtube_labeled_output\( المنطقة الجنوبية ) Youtube Data -  After Cleaning\YouTube Datasets - Al-Baha\Al-Baha Vedio Comments 11_textready.xlsx
  -> Removed missing text rows: 3
  -> Final rows: 111

Processing: C:\Users\ziyad\OneDrive\Desktop